In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Amharic N-gram Language Model Building and Training

This notebook builds and validates Amharic unigram, bigram, trigram, and four-gram language models using the saved preprocessing outputs.

## Inputs and data use

- Processed `train.npy`, `dev.npy`, and `test.npy` arrays
- `vocab.txt` and preprocessing statistics
- Model source code and tests

The model is fitted on `train.npy` only. Smoothing and interpolation choices use complete sentences from `dev.npy`. The test split is not used for fitting or tuning.

## Workflow

1. Inspect the Kaggle environment, attached files, data formats, vocabulary, and preprocessing statistics.
2. Run the unit tests and validate the implementation on a small sentence-aligned training slice.
3. Review the full-counting preflight cell. The full-training cell is run manually only when you are ready.
4. After training, run the analysis cells in order to produce top N-grams, the assigned sentence probabilities, and development-only tuning reports.
5. Run the final packaging cell to validate the model, create downloadable ZIP archives, and print their hashes and links.

All generated files are stored under `/kaggle/working/ngram_results/`. This directory is temporary until downloaded or saved in a Kaggle notebook output/dataset. Checkpoints and intermediate runs are preserved.


In [3]:
from pathlib import Path
import platform
import shutil
import sys

print("Python:", platform.python_version())
print("NumPy:", __import__("numpy").__version__)

for line in Path("/proc/meminfo").read_text().splitlines():
    if line.startswith(("MemTotal:", "MemAvailable:")):
        print(line)

for folder in ("/kaggle/working", "/tmp"):
    if Path(folder).exists():
        d = shutil.disk_usage(folder)
        print(f"{folder}: {d.free / 2**30:.2f} GiB free of {d.total / 2**30:.2f} GiB")

wanted = {
    "ngram_disk.py", "ngram_model.py", "preprocess_lib.py",
    "train_ngram_slice.py", "test_ngram_disk.py",
    "test_ngram_model.py", "test_ngram_slice.py",
    "train.npy", "dev.npy", "test.npy", "vocab.txt", "stats.json",
}
for root in (Path("/kaggle/input"), Path("/kaggle/working")):
    if root.exists():
        for path in root.rglob("*"):
            if path.is_file() and path.name in wanted:
                print(path)

Python: 3.13.15
NumPy: 2.1.3
MemTotal:       32869292 kB
MemAvailable:   31425276 kB
/kaggle/working: 19.50 GiB free of 19.52 GiB
/tmp: 1048.76 GiB free of 8156.88 GiB
/kaggle/input/datasets/naomimeseret/amharic-ngram-processed-full/amharic-ngram-processed-full/dev.npy
/kaggle/input/datasets/naomimeseret/amharic-ngram-processed-full/amharic-ngram-processed-full/stats.json
/kaggle/input/datasets/naomimeseret/amharic-ngram-processed-full/amharic-ngram-processed-full/test.npy
/kaggle/input/datasets/naomimeseret/amharic-ngram-processed-full/amharic-ngram-processed-full/train.npy
/kaggle/input/datasets/naomimeseret/amharic-ngram-processed-full/amharic-ngram-processed-full/vocab.txt
/kaggle/input/datasets/naomimeseret/amharic-ngram-code/src/ngram_disk.py
/kaggle/input/datasets/naomimeseret/amharic-ngram-code/src/ngram_model.py
/kaggle/input/datasets/naomimeseret/amharic-ngram-code/src/preprocess_lib.py
/kaggle/input/datasets/naomimeseret/amharic-ngram-code/src/train_ngram_slice.py
/kaggle/in

In [4]:
from pathlib import Path
import shutil
import subprocess
import sys

input_root = Path("/kaggle/input")
disk_files = [
    p for p in input_root.rglob("ngram_disk.py")
    if p.parent.name == "src"
    and (p.parent / "ngram_model.py").is_file()
    and (p.parent / "preprocess_lib.py").is_file()
]
assert len(disk_files) == 1, f"Expected one code bundle; found: {disk_files}"

CODE_INPUT = disk_files[0].parent.parent
assert all((CODE_INPUT / "tests" / name).is_file() for name in (
    "test_ngram_disk.py", "test_ngram_model.py", "test_ngram_slice.py"
))

CODE_WORK = Path("/kaggle/working/code")
CODE_WORK.mkdir(parents=True, exist_ok=True)
shutil.copytree(CODE_INPUT / "src", CODE_WORK / "src", dirs_exist_ok=True)
shutil.copytree(CODE_INPUT / "tests", CODE_WORK / "tests", dirs_exist_ok=True)
print("Refreshed writable code copy:", CODE_WORK)

result = subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"],
    cwd=CODE_WORK, capture_output=True, text=True,
)
print(result.stdout)
print(result.stderr)
print("Exit code:", result.returncode)
assert result.returncode == 0, "Stop if any Kaggle test fails."

Refreshed writable code copy: /kaggle/working/code

test_bad_sentence_boundaries_are_rejected (test_ngram_disk.DiskBackedNGramTests.test_bad_sentence_boundaries_are_rejected) ... ok
test_disk_counts_match_memory_fit_across_chunks_runs_and_duplicate_keys (test_ngram_disk.DiskBackedNGramTests.test_disk_counts_match_memory_fit_across_chunks_runs_and_duplicate_keys) ... ok
test_interruption_during_merge_rebuilds_incomplete_order_without_duplicates (test_ngram_disk.DiskBackedNGramTests.test_interruption_during_merge_rebuilds_incomplete_order_without_duplicates) ... ok
test_interruption_resumes_from_committed_sentence_boundary (test_ngram_disk.DiskBackedNGramTests.test_interruption_resumes_from_committed_sentence_boundary) ... ok
test_max_tokens_uses_only_complete_sentence_prefix (test_ngram_disk.DiskBackedNGramTests.test_max_tokens_uses_only_complete_sentence_prefix) ... ok
test_add_k_smoothing_uses_prediction_vocabulary (test_ngram_model.NGramModelTests.test_add_k_smoothing_uses_prediction

In [5]:
from pathlib import Path
import hashlib
import json
import numpy as np

DATA_DIR = Path(
    "/kaggle/input/datasets/naomimeseret/"
    "amharic-ngram-processed-full/amharic-ngram-processed-full"
)

paths = {
    split: DATA_DIR / f"{split}.npy"
    for split in ("train", "dev", "test")
}
VOCAB_PATH = DATA_DIR / "vocab.txt"
STATS_PATH = DATA_DIR / "stats.json"

assert all(path.is_file() for path in paths.values()), paths
assert VOCAB_PATH.is_file(), VOCAB_PATH

vocabulary = VOCAB_PATH.read_text(encoding="utf-8").splitlines()
assert vocabulary[:4] == ["<PAD>", "<UNK>", "<s>", "</s>"]
assert len(vocabulary) == 50_000
assert len(set(vocabulary)) == len(vocabulary)

def sha256_file(path, block_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            block = f.read(block_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()

def inspect_array(path, chunk_size=1_000_000):
    array = np.load(path, mmap_mode="r", allow_pickle=False)
    assert array.ndim == 1, f"{path}: expected flat array, got {array.shape}"
    assert array.dtype == np.int32, f"{path}: expected int32, got {array.dtype}"

    minimum, maximum = len(vocabulary), -1
    bos_count = eos_count = pad_count = unk_count = 0
    expected_marker = 2  # sentence stream must begin with <s>
    last_marker_position = None

    for start in range(0, array.size, chunk_size):
        stop = min(start + chunk_size, array.size)
        block = array[start:stop]

        if block.size:
            minimum = min(minimum, int(block.min()))
            maximum = max(maximum, int(block.max()))

        bos_count += int(np.count_nonzero(block == 2))
        eos_count += int(np.count_nonzero(block == 3))
        pad_count += int(np.count_nonzero(block == 0))
        unk_count += int(np.count_nonzero(block == 1))

        marker_positions = np.flatnonzero((block == 2) | (block == 3))
        for relative_pos in marker_positions:
            position = start + int(relative_pos)
            marker = int(block[relative_pos])
            assert marker == expected_marker, (
                f"{path}: unexpected boundary marker {marker} at ID offset {position}"
            )
            if marker == 2 and last_marker_position is not None:
                assert position == last_marker_position + 1, (
                    f"{path}: ordinary IDs occur between sentences at offset {position}"
                )
            last_marker_position = position
            expected_marker = 3 if marker == 2 else 2

    assert array.size > 0, f"{path}: empty split"
    assert minimum >= 0 and maximum < len(vocabulary), (
        f"{path}: ID range [{minimum}, {maximum}] is outside the vocabulary"
    )
    assert pad_count == 0, f"{path}: found {pad_count} <PAD> IDs"
    assert bos_count == eos_count, f"{path}: unpaired <s>/</s> counts"
    assert expected_marker == 2, f"{path}: stream ends without </s>"
    assert last_marker_position == array.size - 1, f"{path}: trailing IDs after final </s>"

    content_ids = array.size - bos_count - eos_count
    return {
        "path": str(path),
        "shape": array.shape,
        "dtype": str(array.dtype),
        "token_ids": int(array.size),
        "file_bytes": path.stat().st_size,
        "id_min": minimum,
        "id_max": maximum,
        "sentences": bos_count,
        "unk_tokens": unk_count,
        "unk_rate_percent": 100 * unk_count / content_ids if content_ids else None,
        "sha256": sha256_file(path),
    }

array_report = {
    split: inspect_array(path)
    for split, path in paths.items()
}

for split, report in array_report.items():
    print(f"\n{split}:")
    print(json.dumps(report, ensure_ascii=False, indent=2))

print("\nVocabulary entries:", len(vocabulary))
print("Vocabulary special IDs:", dict(enumerate(vocabulary[:4])))
print("Vocabulary SHA-256:", sha256_file(VOCAB_PATH))
if STATS_PATH.is_file():
    print("Preprocessing stats:", json.dumps(
        json.loads(STATS_PATH.read_text(encoding="utf-8")),
        ensure_ascii=False,
        indent=2,
    ))


train:
{
  "path": "/kaggle/input/datasets/naomimeseret/amharic-ngram-processed-full/amharic-ngram-processed-full/train.npy",
  "shape": [
    68942614
  ],
  "dtype": "int32",
  "token_ids": 68942614,
  "file_bytes": 275770584,
  "id_min": 1,
  "id_max": 49999,
  "sentences": 4067589,
  "unk_tokens": 8476878,
  "unk_rate_percent": 13.940528589299506,
  "sha256": "c88588aa527108a3376676c355c212dc79136fcfb879f7a4c54796d070dabb42"
}

dev:
{
  "path": "/kaggle/input/datasets/naomimeseret/amharic-ngram-processed-full/amharic-ngram-processed-full/dev.npy",
  "shape": [
    8605850
  ],
  "dtype": "int32",
  "token_ids": 8605850,
  "file_bytes": 34423528,
  "id_min": 1,
  "id_max": 49999,
  "sentences": 508114,
  "unk_tokens": 1067554,
  "unk_rate_percent": 14.06597061092107,
  "sha256": "a06813baf20691bfe4037e5eaaff1447ef83b4febf0412cb84fa096bad29268e"
}

test:
{
  "path": "/kaggle/input/datasets/naomimeseret/amharic-ngram-processed-full/amharic-ngram-processed-full/test.npy",
  "shape": [

In [6]:
from pathlib import Path
import numpy as np

train = np.load(paths["train"], mmap_mode="r", allow_pickle=False)
dev = np.load(paths["dev"], mmap_mode="r", allow_pickle=False)

train_ends = np.flatnonzero(train == 3)
mean_sentence_bytes = (train.size / max(len(train_ends), 1)) * train.dtype.itemsize
estimated_set_bytes = len(train_ends) * (mean_sentence_bytes + 160)
available_ram_bytes = next(
    int(line.split()[1]) * 1024
    for line in Path("/proc/meminfo").read_text().splitlines()
    if line.startswith("MemAvailable:")
)

print(f"Estimated train sentence-set memory: {estimated_set_bytes / 2**30:.2f} GiB")
print(f"Available RAM: {available_ram_bytes / 2**30:.2f} GiB")
assert estimated_set_bytes <= 0.70 * available_ram_bytes, (
    "Overlap set estimate exceeds 70% of available RAM; stop and use a disk-backed index."
)

train_sentences = set()
start = 0
for end in train_ends:
    end = int(end)
    train_sentences.add(train[start:end + 1].tobytes())
    start = end + 1

dev_ends = np.flatnonzero(dev == 3)
matched = 0
start = 0
for end in dev_ends:
    end = int(end)
    matched += train[start:end + 1].tobytes() in train_sentences
    start = end + 1

print("Train sentences indexed:", len(train_sentences))
print("Dev sentences:", len(dev_ends))
print("Dev sentences also present in train:", matched)
print("Exact train/dev overlap percent:", 100 * matched / len(dev_ends))

Estimated train sentence-set memory: 0.86 GiB
Available RAM: 29.87 GiB
Train sentences indexed: 3034391
Dev sentences: 508114
Dev sentences also present in train: 1293
Exact train/dev overlap percent: 0.25447045348091174


In [7]:
from datetime import datetime, timezone
from pathlib import Path
import json
import numpy as np
import resource
import sys

CODE_WORK = Path("/kaggle/working/code")
sys.path.insert(0, str(CODE_WORK))

from src.ngram_disk import DiskBackedNGramTrainer
from src.ngram_model import NGramModel
from src.train_ngram_slice import select_complete_prefix

train = np.load(paths["train"], mmap_mode="r", allow_pickle=False)
selected, sentence_count, inspected = select_complete_prefix(train, 2_000_000)
assert selected.size > 0

memory_model = NGramModel.from_flat_ids(
    selected,
    vocabulary,
    max_order=4,
    max_occurrences=10_000_000,
)

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
smoke_root = Path("/kaggle/working/ngram_results") / f"preflight_{run_id}"

trainer = DiskBackedNGramTrainer(
    paths["train"],
    VOCAB_PATH,
    smoke_root / "work",
    smoke_root / "model",
    max_order=4,
    max_tokens=2_000_000,
    max_sentence_tokens=100,
    scan_chunk_tokens=500_000,
    run_occurrences_limit=2_000_000,
    merge_buffer_size=65_536,
)
result = trainer.run()
disk_model = NGramModel.load(result.model_dir, mmap_mode="r")

assert result.report["complete_sentences"] == sentence_count
assert result.report["source_ids_processed"] == int(selected.size)

for order in range(1, 5):
    np.testing.assert_array_equal(
        disk_model.orders[order].keys, memory_model.orders[order].keys
    )
    np.testing.assert_array_equal(
        disk_model.orders[order].counts, memory_model.orders[order].counts
    )
    np.testing.assert_array_equal(
        disk_model.orders[order].context_keys,
        memory_model.orders[order].context_keys,
    )
    np.testing.assert_array_equal(
        disk_model.orders[order].context_totals,
        memory_model.orders[order].context_totals,
    )

    keys = disk_model.orders[order].keys
    counts = disk_model.orders[order].counts
    targets = keys & np.uint64(0xFFFF)

    assert not np.any(targets == 2), f"Order {order} predicts <s>"
    assert int(counts[targets == 3].sum(dtype=np.uint64)) == sentence_count

    if order > 1:
        for shift in range(1, order):
            context_component = (keys >> np.uint64(16 * shift)) & np.uint64(0xFFFF)
            assert not np.any(context_component == 3), (
                f"Order {order} has </s> inside a context"
            )

target_id = disk_model.word_to_id["<UNK>"]
context_ids = [2, 2, 2]
p = disk_model.prob(target_id, context_ids, method="add_k", k=1.0)
lp = disk_model.logprob(target_id, context_ids, method="add_k", k=1.0)
assert p > 0 and np.isfinite(lp)

print("Disk and in-memory count tables match for all four orders.")
print("Complete IDs:", int(selected.size))
print("Complete sentences:", sentence_count)
print("Inspected IDs:", inspected)
print("Probability:", p, "log probability:", lp)
print("Top five predictions:", disk_model.predict_next("እኔ", k=5))
print("Disk trainer report:")
print(json.dumps(result.report, ensure_ascii=False, indent=2))
print("Smoke model directory:", result.model_dir)
print("Peak RSS KiB:", resource.getrusage(resource.RUSAGE_SELF).ru_maxrss)

Disk and in-memory count tables match for all four orders.
Complete IDs: 1999988
Complete sentences: 138045
Inspected IDs: 2000000
Probability: 0.11885579362167163 log probability: -2.1298443390214166
Top five predictions: [('የምለው', 0.07925281102647806), ('ግን', 0.057127312295973884), ('ነኝ', 0.040805223068552776), ('ደግሞ', 0.02629669931084512), ('ብቻ', 0.018679724338048605)]
Disk trainer report:
{
  "status": "ok",
  "source_ids_processed": 1999988,
  "complete_sentences": 138045,
  "raw_occurrences_by_order": {
    "1": 1861943,
    "2": 1861943,
    "3": 1861943,
    "4": 1861943
  },
  "unique_ngrams_by_order": {
    "1": 46725,
    "2": 778147,
    "3": 1347365,
    "4": 1549808
  },
  "model_array_bytes": 93138656,
  "work_run_bytes_before_cleanup": 67084661,
  "elapsed_seconds": 35.69995899000014,
  "peak_local_occurrence_keys": 2000000,
  "merge_buffer_entries": 65536,
  "cleanup_runs_requested": false
}
Smoke model directory: /kaggle/working/ngram_results/preflight_20261023T193236

In [8]:
from pathlib import Path
import hashlib
import inspect
import json
import shutil
import sys

import numpy as np

DATA_DIR = Path(
    "/kaggle/input/datasets/naomimeseret/"
    "amharic-ngram-processed-full/amharic-ngram-processed-full"
)
TRAIN_PATH = DATA_DIR / "train.npy"
VOCAB_PATH = DATA_DIR / "vocab.txt"

WORK_DIR = Path("/kaggle/working/ngram_results/work")
MODEL_DIR = Path("/kaggle/working/ngram_results/model")

# Match the successful preflight slice settings.
CONFIG = {
    "max_order": 4,
    "scan_chunk_tokens": 1_000_000,
    "run_occurrences_limit": 2_000_000,
    "merge_buffer_size": 65_536,
    "max_tokens": None,
    "max_sentence_tokens": 100,
}

EXPECTED_TRAIN_SHA256 = (
    "c88588aa527108a3376676c355c212dc79136fcfb879f7a4c54796d070dabb42"
)
EXPECTED_VOCAB_SHA256 = (
    "84d8c01602e96fc132bd109d578adaf796b23f1c4aeed696a911b46e7f798e7c"
)

def sha256_file(path, block_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as file:
        while block := file.read(block_size):
            digest.update(block)
    return digest.hexdigest()

# Find the already-uploaded code; this does not copy or modify it.
code_roots = [
    Path("/kaggle/working/code"),
    Path("/kaggle/input/datasets/naomimeseret/amharic-ngram-code"),
]
CODE_ROOT = next(
    (root for root in code_roots if (root / "src" / "ngram_disk.py").is_file()),
    None,
)
if CODE_ROOT is None:
    raise FileNotFoundError("Could not find the uploaded src/ngram_disk.py")

sys.path.insert(0, str(CODE_ROOT))
from src.ngram_disk import DiskBackedNGramTrainer

assert TRAIN_PATH.is_file(), f"Missing training array: {TRAIN_PATH}"
assert VOCAB_PATH.is_file(), f"Missing vocabulary: {VOCAB_PATH}"

train = np.load(TRAIN_PATH, mmap_mode="r", allow_pickle=False)
vocabulary = VOCAB_PATH.read_text(encoding="utf-8").splitlines()

train_hash = sha256_file(TRAIN_PATH)
vocab_hash = sha256_file(VOCAB_PATH)

assert train.ndim == 1 and train.dtype == np.int32, (train.shape, train.dtype)
assert train_hash == EXPECTED_TRAIN_SHA256, train_hash
assert vocab_hash == EXPECTED_VOCAB_SHA256, vocab_hash
assert len(vocabulary) == 50_000, len(vocabulary)
assert vocabulary[:4] == ["<PAD>", "<UNK>", "<s>", "</s>"], vocabulary[:4]

# Current memory and persistent Kaggle working-disk capacity.
meminfo = Path("/proc/meminfo").read_text()
available_kib = int(
    next(line.split()[1] for line in meminfo.splitlines()
         if line.startswith("MemAvailable:"))
)
disk = shutil.disk_usage("/kaggle/working")
gib = 1024 ** 3
available_ram_gib = available_kib / 1024**2
free_disk_gib = disk.free / gib

# Use the same configuration for a new run or resume.
trainer = DiskBackedNGramTrainer(
    TRAIN_PATH, VOCAB_PATH, WORK_DIR, MODEL_DIR, **CONFIG
)

checkpoint = WORK_DIR / "checkpoint.json"
if WORK_DIR.exists() and not checkpoint.exists():
    raise FileExistsError(
        f"{WORK_DIR} exists without a checkpoint. Nothing was changed; "
        "inspect it before deciding how to proceed."
    )

resume = checkpoint.is_file()
if resume:
    saved_state = json.loads(checkpoint.read_text(encoding="utf-8"))
    if saved_state.get("signature") != trainer._signature:
        raise ValueError(
            "Existing checkpoint does not match these inputs/settings. "
            "Do not resume it with this configuration."
        )
elif MODEL_DIR.exists():
    raise FileExistsError(
        f"{MODEL_DIR} already exists without a matching resumable checkpoint. "
        "Nothing was changed; inspect it before proceeding."
    )

print("Code root:", CODE_ROOT)
print("Trainer source:", inspect.getfile(DiskBackedNGramTrainer))
print("Training path:", TRAIN_PATH)
print("Training shape/dtype:", train.shape, train.dtype)
print("Training bytes:", TRAIN_PATH.stat().st_size)
print("Training SHA-256:", train_hash)
print("Vocabulary path:", VOCAB_PATH)
print("Vocabulary entries:", len(vocabulary))
print("Vocabulary SHA-256:", vocab_hash)
print("Configuration:", CONFIG)
print("Work directory:", WORK_DIR, "| exists:", WORK_DIR.exists())
print("Model directory:", MODEL_DIR, "| exists:", MODEL_DIR.exists())
print("Resume checkpoint found:", resume)
print(f"Available RAM: {available_ram_gib:.2f} GiB")
print(f"Free /kaggle/working disk: {free_disk_gib:.2f} GiB")

if available_ram_gib < 4:
    raise RuntimeError("Preflight stopped: less than 4 GiB RAM is available.")
if free_disk_gib < 10:
    raise RuntimeError("Preflight stopped: less than 10 GiB working disk is free.")

print("Preflight passed. No counting was started.")

Code root: /kaggle/working/code
Trainer source: /kaggle/working/code/src/ngram_disk.py
Training path: /kaggle/input/datasets/naomimeseret/amharic-ngram-processed-full/amharic-ngram-processed-full/train.npy
Training shape/dtype: (68942614,) int32
Training bytes: 275770584
Training SHA-256: c88588aa527108a3376676c355c212dc79136fcfb879f7a4c54796d070dabb42
Vocabulary path: /kaggle/input/datasets/naomimeseret/amharic-ngram-processed-full/amharic-ngram-processed-full/vocab.txt
Vocabulary entries: 50000
Vocabulary SHA-256: 84d8c01602e96fc132bd109d578adaf796b23f1c4aeed696a911b46e7f798e7c
Configuration: {'max_order': 4, 'scan_chunk_tokens': 1000000, 'run_occurrences_limit': 2000000, 'merge_buffer_size': 65536, 'max_tokens': None, 'max_sentence_tokens': 100}
Work directory: /kaggle/working/ngram_results/work | exists: False
Model directory: /kaggle/working/ngram_results/model | exists: False
Resume checkpoint found: False
Available RAM: 29.19 GiB
Free /kaggle/working disk: 19.35 GiB
Preflight pa

In [9]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import sys
import threading
import time

import numpy as np

DATA_DIR = Path(
    "/kaggle/input/datasets/naomimeseret/"
    "amharic-ngram-processed-full/amharic-ngram-processed-full"
)
TRAIN_PATH = DATA_DIR / "train.npy"
VOCAB_PATH = DATA_DIR / "vocab.txt"

WORK_DIR = Path("/kaggle/working/ngram_results/work")
MODEL_DIR = Path("/kaggle/working/ngram_results/model")

CONFIG = {
    "max_order": 4,
    "scan_chunk_tokens": 1_000_000,
    "run_occurrences_limit": 2_000_000,
    "merge_buffer_size": 65_536,
    "max_tokens": None,
    "max_sentence_tokens": 100,
}

EXPECTED_TRAIN_SHA256 = (
    "c88588aa527108a3376676c355c212dc79136fcfb879f7a4c54796d070dabb42"
)
EXPECTED_VOCAB_SHA256 = (
    "84d8c01602e96fc132bd109d578adaf796b23f1c4aeed696a911b46e7f798e7c"
)
MIN_FREE_GIB = 10
GIB = 1024 ** 3

CODE_ROOT = Path("/kaggle/working/code")
sys.path.insert(0, str(CODE_ROOT))

from src.ngram_disk import DiskBackedNGramTrainer
from src.ngram_model import BOS, UNK, NGramModel

def sha256_file(path, block_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as file:
        while block := file.read(block_size):
            digest.update(block)
    return digest.hexdigest()

def directory_bytes(path):
    if not path.exists():
        return 0
    total = 0
    for item in path.rglob("*"):
        if item.is_file():
            try:
                total += item.stat().st_size
            except FileNotFoundError:
                pass
    return total

def available_ram_gib():
    meminfo = Path("/proc/meminfo").read_text()
    available_kib = int(
        next(line.split()[1] for line in meminfo.splitlines()
             if line.startswith("MemAvailable:"))
    )
    return available_kib / 1024**2

def peak_rss_gib():
    status = Path("/proc/self/status").read_text()
    high_water_kib = int(
        next(line.split()[1] for line in status.splitlines()
             if line.startswith("VmHWM:"))
    )
    return high_water_kib / 1024**2

# Recheck the same input files before counting.
assert TRAIN_PATH.is_file() and VOCAB_PATH.is_file()
train = np.load(TRAIN_PATH, mmap_mode="r", allow_pickle=False)
vocabulary = VOCAB_PATH.read_text(encoding="utf-8").splitlines()

assert train.ndim == 1 and train.dtype == np.int32
assert train.shape == (68_942_614,)
assert len(vocabulary) == 50_000
assert vocabulary[:4] == ["<PAD>", "<UNK>", "<s>", "</s>"]
assert sha256_file(TRAIN_PATH) == EXPECTED_TRAIN_SHA256
assert sha256_file(VOCAB_PATH) == EXPECTED_VOCAB_SHA256

disk_before = shutil.disk_usage("/kaggle/working")
ram_before = available_ram_gib()
free_before_gib = disk_before.free / GIB
assert free_before_gib >= MIN_FREE_GIB, (
    f"Stopping before training: only {free_before_gib:.2f} GiB is free."
)
assert ram_before >= 4, (
    f"Stopping before training: only {ram_before:.2f} GiB RAM is available."
)

trainer = DiskBackedNGramTrainer(
    TRAIN_PATH, VOCAB_PATH, WORK_DIR, MODEL_DIR, **CONFIG
)
checkpoint_path = WORK_DIR / "checkpoint.json"

if WORK_DIR.exists() and not checkpoint_path.is_file():
    raise FileExistsError(
        f"{WORK_DIR} exists without a checkpoint. Inspect it; nothing was deleted."
    )
resume = checkpoint_path.is_file()

if resume:
    saved_state = json.loads(checkpoint_path.read_text(encoding="utf-8"))
    if saved_state.get("signature") != trainer._signature:
        raise ValueError(
            "Checkpoint inputs/settings differ. Do not resume with this configuration."
        )
elif MODEL_DIR.exists():
    raise FileExistsError(
        f"{MODEL_DIR} already exists without a resumable checkpoint. "
        "Inspect it; nothing was deleted."
    )

print(f"Starting full training; resume={resume}")
print("Training input:", TRAIN_PATH)
print("Vocabulary input:", VOCAB_PATH)
print("Configuration:", CONFIG)
print(f"RAM available at start: {ram_before:.2f} GiB")
print(f"Working disk free at start: {free_before_gib:.2f} GiB")
print("Work directory:", WORK_DIR)
print("Model directory:", MODEL_DIR)

stop_monitor = threading.Event()

def report_progress():
    last_report = None
    while not stop_monitor.wait(30):
        try:
            state = json.loads(checkpoint_path.read_text(encoding="utf-8"))
            progress = (
                state.get("next_offset"),
                state.get("sentences"),
                state.get("next_run_id"),
                tuple(state.get("merge_completed_orders", [])),
            )
            free_gib = shutil.disk_usage("/kaggle/working").free / GIB
            work_gib = directory_bytes(WORK_DIR) / GIB
            model_gib = directory_bytes(MODEL_DIR) / GIB
            if progress != last_report:
                print(
                    f"[progress] committed IDs={progress[0]:,}; "
                    f"sentences={progress[1]:,}; runs={progress[2]}; "
                    f"merged orders={list(progress[3])}; "
                    f"work={work_gib:.2f} GiB; model={model_gib:.2f} GiB; "
                    f"free={free_gib:.2f} GiB",
                    flush=True,
                )
                last_report = progress
            if free_gib < MIN_FREE_GIB:
                print(
                    f"[safety stop] Free disk fell below {MIN_FREE_GIB} GiB. "
                    "Interrupting; committed checkpoints and runs are retained.",
                    flush=True,
                )
                import _thread
                _thread.interrupt_main()
                return
        except (FileNotFoundError, json.JSONDecodeError):
            # A checkpoint is replaced atomically; retry on the next interval.
            continue

monitor = threading.Thread(target=report_progress, daemon=True)
monitor.start()
started = time.perf_counter()

try:
    result = trainer.run(resume=resume, cleanup_runs=False)
finally:
    stop_monitor.set()
    monitor.join(timeout=5)

elapsed = time.perf_counter() - started
disk_after = shutil.disk_usage("/kaggle/working")
loaded = NGramModel.load(MODEL_DIR, mmap_mode="r")
metadata = json.loads((MODEL_DIR / "metadata.json").read_text(encoding="utf-8"))

# Every prediction event is an ordinary token or </s>; <s> is context only.
expected_events = len(train) - int(np.count_nonzero(train == BOS))
assert result.report["source_ids_processed"] == len(train)
assert result.report["complete_sentences"] == int(np.count_nonzero(train == BOS))

for order in range(1, 5):
    expected_total = expected_events
    actual_total = loaded.total_occurrences(order)
    assert actual_total == expected_total, (order, actual_total, expected_total)
    assert loaded.unique_ngrams(order) == int(
        metadata["unique_ngrams_by_order"][str(order)]
    )
    assert actual_total == int(metadata["raw_occurrences_by_order"][str(order)])

# Exercise saved-model queries after reload.
query_probability = loaded.prob(UNK, [BOS], method="add_k", k=1.0)
query_logprob = loaded.logprob(UNK, [BOS], method="add_k", k=1.0)
query_context = (
    ["ኢትዮጵያ"]
    if "ኢትዮጵያ" in loaded.word_to_id
    else ["<UNK>"]
)
query_predictions = loaded.predict_next(query_context, k=5)

print("\nFULL TRAINING COMPLETE")
print("Trainer report:", json.dumps(result.report, ensure_ascii=False, indent=2))
print("Elapsed seconds:", round(elapsed, 2))
print(f"Peak process RSS: {peak_rss_gib():.2f} GiB")
print(f"Working disk used by run: "
      f"{(directory_bytes(WORK_DIR) + directory_bytes(MODEL_DIR)) / GIB:.2f} GiB")
print(f"Working disk free after run: {disk_after.free / GIB:.2f} GiB")
print("Verified expected occurrences per order:", expected_events)
print("Verified unique N-grams per order:", metadata["unique_ngrams_by_order"])
print("Reloaded add-k probability:", query_probability)
print("Reloaded add-k log probability:", query_logprob)
print("Reloaded top-five predictions:", query_predictions)
print("Saved model directory:", MODEL_DIR)
print("Checkpoint and intermediate runs retained:", WORK_DIR)

# Persist the run summary beside the model so it remains available for packaging.
REPORTS_DIR = Path("/kaggle/working/ngram_results/reports")
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
training_report = {
    "status": "ok",
    "input_train_path": str(TRAIN_PATH),
    "input_train_sha256": sha256_file(TRAIN_PATH),
    "input_vocab_path": str(VOCAB_PATH),
    "input_vocab_sha256": sha256_file(VOCAB_PATH),
    "elapsed_seconds": elapsed,
    "peak_rss_gib": peak_rss_gib(),
    "working_disk_used_gib": (directory_bytes(WORK_DIR) + directory_bytes(MODEL_DIR)) / GIB,
    "trainer_report": result.report,
    "model_metadata": metadata,
    "query_checks": {
        "add_k_probability": query_probability,
        "add_k_log_probability": query_logprob,
        "predict_next_top_five": query_predictions,
    },
}
training_report_path = REPORTS_DIR / "training_report.json"
training_report_path.write_text(
    json.dumps(training_report, ensure_ascii=False, indent=2, allow_nan=False) + "\n",
    encoding="utf-8",
)
print("Saved training report:", training_report_path, training_report_path.stat().st_size, "bytes")

Starting full training; resume=False
Training input: /kaggle/input/datasets/naomimeseret/amharic-ngram-processed-full/amharic-ngram-processed-full/train.npy
Vocabulary input: /kaggle/input/datasets/naomimeseret/amharic-ngram-processed-full/amharic-ngram-processed-full/vocab.txt
Configuration: {'max_order': 4, 'scan_chunk_tokens': 1000000, 'run_occurrences_limit': 2000000, 'merge_buffer_size': 65536, 'max_tokens': None, 'max_sentence_tokens': 100}
RAM available at start: 29.22 GiB
Working disk free at start: 19.35 GiB
Work directory: /kaggle/working/ngram_results/work
Model directory: /kaggle/working/ngram_results/model
[progress] committed IDs=2,148,203; sentences=148,243; runs=4; merged orders=[]; work=0.07 GiB; model=0.00 GiB; free=19.29 GiB
[progress] committed IDs=4,296,253; sentences=296,332; runs=8; merged orders=[]; work=0.13 GiB; model=0.00 GiB; free=19.22 GiB
[progress] committed IDs=6,445,385; sentences=445,530; runs=12; merged orders=[]; work=0.20 GiB; model=0.00 GiB; free=1

In [10]:
from pathlib import Path
from datetime import datetime, timezone
import csv
import json

import numpy as np

MODEL_DIR = Path("/kaggle/working/ngram_results/model")
RESULTS_ROOT = Path("/kaggle/working/ngram_results")
OUTPUT_DIR = RESULTS_ROOT / "analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CODE_ROOT = Path("/kaggle/working/code")
if str(CODE_ROOT) not in __import__("sys").path:
    __import__("sys").path.insert(0, str(CODE_ROOT))

from src.ngram_model import NGramModel

assert (MODEL_DIR / "metadata.json").is_file()
model = NGramModel.load(MODEL_DIR, mmap_mode="r")
metadata = json.loads((MODEL_DIR / "metadata.json").read_text(encoding="utf-8"))

assert model.max_order == 4
assert all(order in model.orders for order in range(1, 5))

TOP_K = 10
CHUNK_SIZE = 1_000_000
MIN_BIGRAM_COUNT = 20
ADD_K = 1.0

# Word-only outputs omit structural tokens and any token containing <NUM>.
excluded_words = {"<PAD>", "<UNK>", "<s>", "</s>", "<NUM>"}
excluded_ids = {
    token_id
    for token_id, token in enumerate(model.vocabulary)
    if token in excluded_words or "<NUM>" in token
}

def unpack_key(key, order):
    return tuple(
        (int(key) >> (16 * shift)) & 0xFFFF
        for shift in range(order - 1, -1, -1)
    )

def top_candidates(order, *, k=10, word_only=False, min_count=None):
    """Find top rows by scanning mapped count tables in bounded chunks."""
    counts = model.orders[order].counts
    keys = model.orders[order].keys
    candidates = []

    for start in range(0, len(counts), CHUNK_SIZE):
        stop = min(start + CHUNK_SIZE, len(counts))
        count_chunk = counts[start:stop]
        key_chunk = keys[start:stop]

        valid = np.ones(len(count_chunk), dtype=bool)
        if min_count is not None:
            valid &= count_chunk >= min_count

        if word_only and excluded_ids:
            for shift in range(order - 1, -1, -1):
                token_ids = (key_chunk >> np.uint64(16 * shift)) & np.uint64(0xFFFF)
                valid &= ~np.isin(token_ids, list(excluded_ids))

        local_indexes = np.flatnonzero(valid)
        if len(local_indexes) > k:
            local_counts = count_chunk[local_indexes]
            keep = np.argpartition(local_counts, -k)[-k:]
            local_indexes = local_indexes[keep]

        for local_index in local_indexes:
            candidates.append((
                int(count_chunk[local_index]),
                int(key_chunk[local_index]),
            ))

    # Count descending; token strings break ties deterministically.
    candidates.sort(
        key=lambda item: (
            -item[0],
            tuple(model.vocabulary[i] for i in unpack_key(item[1], order)),
        )
    )
    return candidates[:k]

def write_top_csv(path, order, rows):
    columns = ["rank", "ngram", *[f"token_{i}" for i in range(1, order + 1)], "count"]
    with path.open("w", encoding="utf-8-sig", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=columns)
        writer.writeheader()
        for rank, (count, packed_key) in enumerate(rows, start=1):
            tokens = [model.vocabulary[i] for i in unpack_key(packed_key, order)]
            writer.writerow({
                "rank": rank,
                "ngram": " ".join(tokens),
                **{f"token_{i}": token for i, token in enumerate(tokens, start=1)},
                "count": count,
            })

top_report = {}
for order in range(1, 5):
    all_rows = top_candidates(order, k=TOP_K)
    word_rows = top_candidates(order, k=TOP_K, word_only=True)
    assert all(
        model.vocabulary[token_id] not in excluded_words
        and "<NUM>" not in model.vocabulary[token_id]
        for _, packed_key in word_rows
        for token_id in unpack_key(packed_key, order)
    ), f"Order {order} word-only results contain an excluded token"

    order_name = {1: "unigram", 2: "bigram", 3: "trigram", 4: "fourgram"}[order]
    all_path = OUTPUT_DIR / f"{order_name}_top10_all_tokens.csv"
    word_path = OUTPUT_DIR / f"{order_name}_top10_words_only.csv"
    write_top_csv(all_path, order, all_rows)
    write_top_csv(word_path, order, word_rows)

    top_report[str(order)] = {
        "all_tokens": [
            {
                "tokens": [model.vocabulary[i] for i in unpack_key(key, order)],
                "count": count,
            }
            for count, key in all_rows
        ],
        "word_only": [
            {
                "tokens": [model.vocabulary[i] for i in unpack_key(key, order)],
                "count": count,
            }
            for count, key in word_rows
        ],
    }

    print(f"\nORDER {order}: TOP {TOP_K}, ALL TOKENS")
    for rank, (count, key) in enumerate(all_rows, start=1):
        tokens = [model.vocabulary[i] for i in unpack_key(key, order)]
        print(f"{rank:2}. {' '.join(tokens)}  count={count:,}")

    print(f"ORDER {order}: TOP {TOP_K}, WORD ONLY")
    for rank, (count, key) in enumerate(word_rows, start=1):
        tokens = [model.vocabulary[i] for i in unpack_key(key, order)]
        print(f"{rank:2}. {' '.join(tokens)}  count={count:,}")

# Rank ordinary-word bigrams by occurrence count, requiring count >= 20.
bigram_rows = top_candidates(
    2,
    k=5,
    word_only=True,
    min_count=MIN_BIGRAM_COUNT,
)

if len(bigram_rows) < 5:
    raise RuntimeError(
        f"Found only {len(bigram_rows)} eligible bigrams with count "
        f">={MIN_BIGRAM_COUNT}; expected five."
    )

bigram_report = []
for rank, (bigram_count, packed_key) in enumerate(bigram_rows, start=1):
    left_id, right_id = unpack_key(packed_key, 2)
    left_word = model.vocabulary[left_id]
    right_word = model.vocabulary[right_id]
    context_count = model.orders[2].context_count([left_id])

    mle_probability = bigram_count / context_count
    add_k_probability = model.prob(
        right_id, [left_id], method="add_k", k=ADD_K
    )

    bigram_report.append({
        "rank": rank,
        "word_1": left_word,
        "word_2": right_word,
        "bigram_count": bigram_count,
        "context_count": context_count,
        "mle_probability": mle_probability,
        "add_k": ADD_K,
        "add_k_probability": add_k_probability,
        "prediction_vocabulary_size": len(model.allowed_outcomes),
    })

print(f"\nTOP FIVE WORD-ONLY BIGRAMS (count >= {MIN_BIGRAM_COUNT})")
print("Add-k uses denominator: context_count + k × prediction_vocabulary_size")
for row in bigram_report:
    print(
        f"{row['rank']}. {row['word_1']} {row['word_2']} | "
        f"bigram_count={row['bigram_count']:,} | "
        f"context_count={row['context_count']:,} | "
        f"MLE={row['mle_probability']:.12g} | "
        f"add-{ADD_K:g}={row['add_k_probability']:.12g}"
    )

(OUTPUT_DIR / "bigram_conditional_probabilities.json").write_text(
    json.dumps(bigram_report, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
with (OUTPUT_DIR / "bigram_conditional_probabilities.csv").open(
    "w", encoding="utf-8-sig", newline=""
) as file:
    fields = list(bigram_report[0])
    writer = csv.DictWriter(file, fieldnames=fields)
    writer.writeheader()
    writer.writerows(bigram_report)

report = {
    "model_directory": str(MODEL_DIR),
    "model_max_order": model.max_order,
    "source_train_path": metadata.get("source_train_path"),
    "unique_ngrams_by_order": metadata.get("unique_ngrams_by_order"),
    "top_ngrams": top_report,
    "bigram_probability_settings": {
        "candidate_minimum_count": MIN_BIGRAM_COUNT,
        "candidate_words_exclude": sorted(excluded_words),
        "candidate_token_substring_excluded": "<NUM>",
        "smoothing": "add-k",
        "k": ADD_K,
        "prediction_vocabulary_size": len(model.allowed_outcomes),
        "prediction_outcomes": (
            "all vocabulary IDs except <PAD> and <s>; includes <UNK> and </s>"
        ),
    },
    "bigram_conditional_probabilities": bigram_report,
}
(OUTPUT_DIR / "top10_and_bigram_report.json").write_text(
    json.dumps(report, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

print("\nOUTPUT DIRECTORY:", OUTPUT_DIR)
print("OUTPUT FILES:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(f"{path.name}: {path.stat().st_size:,} bytes")


ORDER 1: TOP 10, ALL TOKENS
 1. <UNK>  count=8,476,878
 2. </s>  count=4,067,589
 3. ነው  count=921,688
 4. <NUM>  count=887,452
 5. ላይ  count=594,173
 6. ውስጥ  count=273,656
 7. ወደ  count=260,934
 8. እና  count=246,281
 9. ጋር  count=229,818
10. ነበር  count=217,891
ORDER 1: TOP 10, WORD ONLY
 1. ነው  count=921,688
 2. ላይ  count=594,173
 3. ውስጥ  count=273,656
 4. ወደ  count=260,934
 5. እና  count=246,281
 6. ጋር  count=229,818
 7. ነበር  count=217,891
 8. ግን  count=214,337
 9. ጊዜ  count=193,009
10. ነገር  count=171,940

ORDER 2: TOP 10, ALL TOKENS
 1. <UNK> <UNK>  count=1,491,423
 2. <UNK> </s>  count=630,379
 3. ነው </s>  count=599,837
 4. <s> <UNK>  count=515,819
 5. <NUM> <NUM>  count=252,742
 6. ነበር </s>  count=163,859
 7. <UNK> ነው  count=115,304
 8. ላይ <UNK>  count=102,680
 9. ናቸው </s>  count=85,843
10. አ ም  count=81,929
ORDER 2: TOP 10, WORD ONLY
 1. አ ም  count=81,929
 2. ነገር ግን  count=35,269
 3. ዶ ር  count=28,055
 4. ብቻ ሳይሆን  count=25,946
 5. ማለት ነው  count=25,588
 6. አዲስ አበባ  count=24,541
 7

In [ ]:
# Add empirical joint probabilities while preserving the original count-only CSVs.
# For each order, P(ngram) = count(ngram) / all prediction events of that order.
# The denominator includes EOS and UNK events; the word-only files only filter rows.
probability_denominators = {
    order: int(metadata["raw_occurrences_by_order"][str(order)])
    for order in range(1, 5)
}
assert set(probability_denominators) == {1, 2, 3, 4}
assert len(set(probability_denominators.values())) == 1

for order_name in ("unigram", "bigram", "trigram", "fourgram"):
    order = {"unigram": 1, "bigram": 2, "trigram": 3, "fourgram": 4}[order_name]
    denominator = probability_denominators[order]
    for token_filter in ("all_tokens", "words_only"):
        source_path = OUTPUT_DIR / f"{order_name}_top10_{token_filter}.csv"
        output_path = OUTPUT_DIR / (
            f"{order_name}_top10_{token_filter}_with_probability.csv"
        )
        with source_path.open(encoding="utf-8-sig", newline="") as source_file:
            reader = csv.DictReader(source_file)
            rows = list(reader)
            fieldnames = list(reader.fieldnames or [])
        if "count" not in fieldnames or len(rows) != TOP_K:
            raise ValueError(f"Unexpected top-10 CSV format: {source_path}")
        output_fields = fieldnames + ["total_occurrences", "joint_probability"]
        with output_path.open("w", encoding="utf-8-sig", newline="") as output_file:
            writer = csv.DictWriter(output_file, fieldnames=output_fields)
            writer.writeheader()
            for row in rows:
                count = int(row["count"])
                row["total_occurrences"] = denominator
                row["joint_probability"] = count / denominator
                writer.writerow(row)
        print(
            f"{output_path.name}: denominator={denominator:,}; "
            f"top probability={int(rows[0]['count']) / denominator:.12g}"
        )

print("Probability type: empirical joint n-gram probability, not conditional probability.")
print("Original count-only top-10 CSVs were preserved.")


In [12]:
from pathlib import Path
from datetime import datetime, timezone
import csv
import json
import math
import sys
import time

import numpy as np

CODE_ROOT = Path("/kaggle/working/code")
if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

from src.ngram_model import BOS, EOS, NGramModel
from src.preprocess_lib import normalize, tokenize

TARGET_MODEL_DIR = Path("/kaggle/working/ngram_results/model")
RESULTS_ROOT = Path("/kaggle/working/ngram_results")
SENTENCE = "ኢትዮጵያ በአፍሪካ ቀንድ ትገኛለች።"
ADD_K = 1.0

assert (TARGET_MODEL_DIR / "metadata.json").is_file(), (
    f"Missing saved full model: {TARGET_MODEL_DIR}"
)
prob_model = NGramModel.load(TARGET_MODEL_DIR, mmap_mode="r")
assert prob_model.max_order == 4

normalized_text = normalize(SENTENCE)
tokens = tokenize(SENTENCE)
unknown_tokens = [token for token in tokens if token not in prob_model.word_to_id]
token_ids = [prob_model.word_to_id.get(token, 1) for token in tokens]

# Training counts each word and EOS as a prediction; BOS appears only in the
# order-specific left context. Every order starts with its own BOS padding.
methods = [("mle", None), ("add_k", ADD_K)]
result_rows = []
for order in range(1, prob_model.max_order + 1):
    framed = [BOS] * (order - 1) + token_ids + [EOS]
    for method, k in methods:
        log_probability = 0.0
        direct_probability = 1.0
        zero_events = []
        for target_index in range(order - 1, len(framed)):
            context = framed[target_index - order + 1:target_index]
            target_id = framed[target_index]
            kwargs = {"method": method}
            if method == "add_k":
                kwargs["k"] = k
            conditional = prob_model.prob(target_id, context, **kwargs)
            conditional_log = prob_model.logprob(target_id, context, **kwargs)
            direct_probability *= conditional
            if conditional == 0.0:
                log_probability = float("-inf")
                zero_events.append({
                    "context": [prob_model.vocabulary[i] for i in context],
                    "target": prob_model.vocabulary[target_id],
                    "target_id": int(target_id),
                })
            elif math.isfinite(log_probability):
                log_probability += conditional_log

        if log_probability == float("-inf"):
            probability = 0.0
        else:
            probability = math.exp(log_probability)
        underflow = probability == 0.0 and math.isfinite(log_probability)
        direct_agrees = (
            direct_probability == probability
            if probability == 0.0
            else math.isclose(direct_probability, probability, rel_tol=1e-12, abs_tol=0.0)
        )
        result_rows.append({
            "order": order,
            "method": method,
            "k": k,
            "log_probability": log_probability,
            "probability": probability,
            "zero_probability_events": zero_events,
            "probability_underflow": underflow,
            "direct_check_agrees": direct_agrees,
        })

report = {
    "sentence": SENTENCE,
    "normalized_text": normalized_text,
    "tokens": tokens,
    "unknown_tokens": unknown_tokens,
    "unknown_token_count": len(unknown_tokens),
    "model_directory": str(TARGET_MODEL_DIR),
    "model_max_order": prob_model.max_order,
    "boundary_convention": (
        "Each order prepends order-1 <s> contexts and predicts every token plus </s>; "
        "<s> is not predicted."
    ),
    "smoothing": {
        "method": "add_k",
        "k": ADD_K,
        "prediction_vocabulary_size": len(prob_model.allowed_outcomes),
        "prediction_outcomes": "all IDs except <PAD> and <s>; includes <UNK> and </s>",
    },
    "results": result_rows,
}

# JSON has no standard representation for infinities; encode them as strings.
def make_json_safe(value):
    if isinstance(value, dict):
        return {str(key): make_json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [make_json_safe(item) for item in value]
    if isinstance(value, (float, np.floating)):
        number = float(value)
        if math.isnan(number):
            return "NaN"
        if math.isinf(number):
            return "-Infinity" if number < 0 else "Infinity"
        return number
    if isinstance(value, np.integer):
        return int(value)
    return value

output_dir = RESULTS_ROOT / "analysis"
output_dir.mkdir(parents=True, exist_ok=True)
csv_path = output_dir / "target_sentence_probabilities.csv"
json_path = output_dir / "target_sentence_probabilities.json"
columns = [
    "sentence", "normalized_text", "tokens_json", "unknown_tokens_json",
    "unknown_token_count", "order", "method", "k", "log_probability",
    "probability", "probability_underflow", "zero_probability_event_count",
    "direct_check_agrees",
]
with csv_path.open("w", encoding="utf-8-sig", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=columns)
    writer.writeheader()
    for row in result_rows:
        writer.writerow({
            "sentence": SENTENCE,
            "normalized_text": normalized_text,
            "tokens_json": json.dumps(tokens, ensure_ascii=False),
            "unknown_tokens_json": json.dumps(unknown_tokens, ensure_ascii=False),
            "unknown_token_count": len(unknown_tokens),
            "order": row["order"],
            "method": row["method"],
            "k": "" if row["k"] is None else row["k"],
            "log_probability": (
                "-Infinity" if row["log_probability"] == float("-inf")
                else row["log_probability"]
            ),
            "probability": row["probability"],
            "probability_underflow": row["probability_underflow"],
            "zero_probability_event_count": len(row["zero_probability_events"]),
            "direct_check_agrees": row["direct_check_agrees"],
        })
json_path.write_text(
    json.dumps(make_json_safe(report), ensure_ascii=False, indent=2, allow_nan=False) + "\n",
    encoding="utf-8",
)

print("Normalized text:", normalized_text)
print("Tokens:", tokens)
print("Unknown tokens:", unknown_tokens)
print("Unknown-token count:", len(unknown_tokens))
print("Smoothing: add-k with k=", ADD_K)
for row in result_rows:
    logp = row["log_probability"]
    print(
        f"order={row['order']} method={row['method']} k={row['k']} "
        f"logP={logp} P={row['probability']:.17g} "
        f"zero_events={len(row['zero_probability_events'])} "
        f"underflow={row['probability_underflow']} "
        f"direct_check={row['direct_check_agrees']}"
    )
print("\nSaved files:")
for path in (csv_path, json_path):
    print(f"{path}: exists={path.is_file()}, bytes={path.stat().st_size:,}")


Normalized text: ኢትዮጵያ በአፍሪካ ቀንድ ትገኛለች።
Tokens: ['ኢትዮጵያ', 'በአፍሪካ', 'ቀንድ', 'ትገኛለች']
Unknown tokens: []
Unknown-token count: 0
Smoothing: add-k with k= 1.0
order=1 method=mle k=None logP=-38.66856742619231 P=1.6086239992341189e-17 zero_events=0 underflow=False direct_check=True
order=1 method=add_k k=1.0 logP=-38.671436879600364 P=1.6040147438017213e-17 zero_events=0 underflow=False direct_check=True
order=2 method=mle k=None logP=-inf P=0 zero_events=1 underflow=False direct_check=True
order=2 method=add_k k=1.0 logP=-30.440644202405963 P=6.0227688745145818e-14 zero_events=0 underflow=False direct_check=True
order=3 method=mle k=None logP=-inf P=0 zero_events=2 underflow=False direct_check=True
order=3 method=add_k k=1.0 logP=-40.718678242066844 P=2.0706310829957191e-18 zero_events=0 underflow=False direct_check=True
order=4 method=mle k=None logP=-inf P=0 zero_events=2 underflow=False direct_check=True
order=4 method=add_k k=1.0 logP=-41.420446569389895 P=1.02642829842658e-18 zero_even

In [12]:
import json
import math
import numpy as np

def make_json_safe(value):
    if isinstance(value, dict):
        return {str(key): make_json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [make_json_safe(item) for item in value]
    if isinstance(value, (float, np.floating)):
        if math.isnan(float(value)):
            return "NaN"
        if math.isinf(float(value)):
            return "-Infinity" if value < 0 else "Infinity"
        return float(value)
    if isinstance(value, np.integer):
        return int(value)
    return value

assert "report" in globals() and "output_dir" in globals()
json_path = output_dir / "target_sentence_probabilities.json"
json_path.write_text(
    json.dumps(make_json_safe(report), ensure_ascii=False, indent=2, allow_nan=False) + "\n",
    encoding="utf-8",
)

print("Normalized text:", report["normalized_text"])
print("Tokens:", report["tokens"])
print("Unknown tokens:", report["unknown_tokens"])
print("Unknown-token count:", report["unknown_token_count"])

for result in report["results"]:
    logp = result["log_probability"]
    print(
        f"order={result['order']} method={result['method']} "
        f"k={result['k']} logP={logp} "
        f"P={result['probability']:.17g} "
        f"zero_events={len(result['zero_probability_events'])} "
        f"underflow={result['probability_underflow']} "
        f"direct_check={result['direct_check_agrees']}"
    )

print("\nSaved files:")
for path in (
    output_dir / "target_sentence_probabilities.csv",
    json_path,
):
    print(f"{path}: exists={path.is_file()}, bytes={path.stat().st_size:,}")

Normalized text: ኢትዮጵያ በአፍሪካ ቀንድ ትገኛለች።
Tokens: ['ኢትዮጵያ', 'በአፍሪካ', 'ቀንድ', 'ትገኛለች']
Unknown tokens: []
Unknown-token count: 0
order=1 method=mle k=None logP=-38.66856742619231 P=1.6086239992341189e-17 zero_events=0 underflow=False direct_check=True
order=1 method=add_k k=1.0 logP=-38.671436879600364 P=1.6040147438017213e-17 zero_events=0 underflow=False direct_check=True
order=2 method=mle k=None logP=-inf P=0 zero_events=1 underflow=False direct_check=True
order=2 method=add_k k=1.0 logP=-30.440644202405963 P=6.0227688745145818e-14 zero_events=0 underflow=False direct_check=True
order=3 method=mle k=None logP=-inf P=0 zero_events=2 underflow=False direct_check=True
order=3 method=add_k k=1.0 logP=-40.718678242066844 P=2.0706310829957191e-18 zero_events=0 underflow=False direct_check=True
order=4 method=mle k=None logP=-inf P=0 zero_events=2 underflow=False direct_check=True
order=4 method=add_k k=1.0 logP=-41.420446569389895 P=1.02642829842658e-18 zero_events=0 underflow=False direct_c

In [13]:
from pathlib import Path
from datetime import datetime, timezone
import csv
import itertools
import json
import math
import sys
import time

import numpy as np

CODE_ROOT = Path("/kaggle/working/code")
if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

from src.ngram_model import BOS, EOS, PAD, NGramModel

MODEL_DIR = Path("/kaggle/working/ngram_results/model")
DEV_PATH = Path(
    "/kaggle/input/datasets/naomimeseret/"
    "amharic-ngram-processed-full/amharic-ngram-processed-full/dev.npy"
)
RESULTS_ROOT = Path("/kaggle/working/ngram_results")

SEED = 42
SAMPLE_SENTENCES = 5_000
ADD_K_GRID = [0.01, 0.1, 0.5, 1.0, 2.0, 5.0]
WEIGHT_STEP = 0.25
NORMALIZATION_TOLERANCE = 1e-6

assert (MODEL_DIR / "metadata.json").is_file()
assert DEV_PATH.is_file()

model = NGramModel.load(MODEL_DIR, mmap_mode="r")
dev = np.load(DEV_PATH, mmap_mode="r", allow_pickle=False)

assert model.max_order == 4
assert dev.ndim == 1 and dev.dtype == np.int32

# Locate complete sentence boundaries; sample sentence spans, not arbitrary IDs.
starts = np.flatnonzero(dev == BOS)
ends = np.flatnonzero(dev == EOS)
assert len(starts) == len(ends), (
    f"Boundary mismatch: {len(starts)} starts vs {len(ends)} ends"
)
assert np.all(ends > starts), "Found an end marker before its sentence start"

sample_size = min(SAMPLE_SENTENCES, len(starts))
rng = np.random.default_rng(SEED)
sample_indexes = np.sort(
    rng.choice(len(starts), size=sample_size, replace=False)
)

# Store sufficient statistics and component probabilities for the sampled
# development prediction events. Histories reset at every sentence.
ngram_counts = {order: [] for order in range(1, 5)}
context_totals = {order: [] for order in range(1, 5)}
interpolation_components = {order: [] for order in range(1, 5)}
representative_histories = []

for sentence_index in sample_indexes:
    start = int(starts[sentence_index])
    end = int(ends[sentence_index])
    sentence = np.asarray(dev[start:end + 1], dtype=np.int32)

    if len(sentence) < 2 or sentence[0] != BOS or sentence[-1] != EOS:
        raise ValueError(f"Malformed sampled sentence at index {sentence_index}")
    if np.any(sentence[1:-1] == BOS) or np.any(sentence[1:-1] == EOS):
        raise ValueError(f"Nested boundary marker in sentence {sentence_index}")
    if np.any(sentence[1:-1] == PAD):
        raise ValueError(f"<PAD> inside sentence {sentence_index}")

    history = [BOS, BOS, BOS]
    for target_raw in sentence[1:]:  # ordinary tokens and final EOS
        target = int(target_raw)
        full_context = tuple(history[-3:])
        if (len(representative_histories) < 5
                and full_context not in representative_histories):
            representative_histories.append(full_context)

        for order in range(1, 5):
            context = history[-(order - 1):] if order > 1 else []
            order_counts = model.orders[order]
            count = order_counts.count([*context, target])

            denominator = (
                model.total_occurrences(1)
                if order == 1
                else order_counts.context_count(context)
            )

            ngram_counts[order].append(count)
            context_totals[order].append(denominator)

            # Match the model's interpolation behavior: use MLE when the
            # component context exists, otherwise use its backoff probability.
            if order == 1 or denominator > 0:
                component_probability = count / denominator if denominator else 0.0
            else:
                component_probability = model.prob(
                    target, context, method="backoff"
                )
            interpolation_components[order].append(component_probability)

        history.append(target)

if len(representative_histories) < 5:
    raise RuntimeError("Could not collect five distinct development contexts")

ngram_counts = {
    order: np.asarray(values, dtype=np.float64)
    for order, values in ngram_counts.items()
}
context_totals = {
    order: np.asarray(values, dtype=np.float64)
    for order, values in context_totals.items()
}
interpolation_components = {
    order: np.asarray(values, dtype=np.float64)
    for order, values in interpolation_components.items()
}

event_count = len(ngram_counts[1])
assert event_count > 0
assert all(len(ngram_counts[o]) == event_count for o in range(1, 5))

# Add-k tuning is performed separately for each model order.
add_k_grid_rows = []
best_k = {}

for order in range(1, 5):
    counts = ngram_counts[order]
    totals = context_totals[order]
    best_row = None

    for k in ADD_K_GRID:
        event_logprobs = np.log(counts + k) - np.log(
            totals + k * len(model.allowed_outcomes)
        )
        mean_logprob = float(event_logprobs.mean())
        row = {
            "order": order,
            "k": k,
            "mean_dev_log_probability": mean_logprob,
            "prediction_vocabulary_size": len(model.allowed_outcomes),
            "events": event_count,
        }
        add_k_grid_rows.append(row)

        # Grid is ascending, so ties keep the smaller k.
        if best_row is None or mean_logprob > best_row["mean_dev_log_probability"]:
            best_row = row

    best_k[order] = best_row

# Enumerate all four-order weight vectors in increments of 0.25 that sum to 1.
units = round(1.0 / WEIGHT_STEP)
weight_candidates = []
for first in range(units + 1):
    for second in range(units - first + 1):
        for third in range(units - first - second + 1):
            fourth = units - first - second - third
            weight_candidates.append(tuple(
                value / units for value in (first, second, third, fourth)
            ))

component_matrix = np.column_stack([
    interpolation_components[order] for order in range(1, 5)
])

interpolation_grid_rows = []
best_interpolation = None
for weights in weight_candidates:
    mixed = component_matrix @ np.asarray(weights, dtype=np.float64)
    mean_logprob = (
        float("-inf")
        if np.any(mixed <= 0)
        else float(np.log(mixed).mean())
    )
    row = {
        "weights_unigram_bigram_trigram_fourgram": list(weights),
        "mean_dev_log_probability": mean_logprob,
        "events": event_count,
    }
    interpolation_grid_rows.append(row)

    if (best_interpolation is None
            or mean_logprob > best_interpolation["mean_dev_log_probability"]):
        best_interpolation = row

if best_interpolation is None or not math.isfinite(
    best_interpolation["mean_dev_log_probability"]
):
    raise RuntimeError("No finite interpolation candidate was found")

# Verify normalized distributions for five real contexts. For each order,
# use the suffix of the context with the required length. Also check the
# selected four-order interpolation distribution on all five full contexts.
normalization_rows = []
for context_number, history in enumerate(representative_histories, start=1):
    for order in range(1, 5):
        context = list(history[-(order - 1):]) if order > 1 else []
        k = best_k[order]["k"]
        probability_sum = math.fsum(
            model.prob(token_id, context, method="add_k", k=k)
            for token_id in model.allowed_outcomes
        )
        row = {
            "context_number": context_number,
            "method": "add_k",
            "order": order,
            "context": [model.vocabulary[token_id] for token_id in context],
            "parameter": {"k": k},
            "probability_sum": probability_sum,
            "absolute_error_from_one": abs(probability_sum - 1.0),
        }
        assert row["absolute_error_from_one"] <= NORMALIZATION_TOLERANCE, row
        normalization_rows.append(row)

    interpolation_weights = best_interpolation[
        "weights_unigram_bigram_trigram_fourgram"
    ]
    context = list(history)
    probability_sum = math.fsum(
        model.prob(
            token_id,
            context,
            method="interpolation",
            weights=interpolation_weights,
        )
        for token_id in model.allowed_outcomes
    )
    row = {
        "context_number": context_number,
        "method": "interpolation",
        "order": 4,
        "context": [model.vocabulary[token_id] for token_id in context],
        "parameter": {"weights": interpolation_weights},
        "probability_sum": probability_sum,
        "absolute_error_from_one": abs(probability_sum - 1.0),
    }
    assert row["absolute_error_from_one"] <= NORMALIZATION_TOLERANCE, row
    normalization_rows.append(row)

# Keep the latest reproducible analysis products together for packaging.
output_dir = RESULTS_ROOT / "analysis"
output_dir.mkdir(parents=True, exist_ok=True)

def write_csv(path, rows):
    if not rows:
        raise ValueError(f"No rows for {path.name}")
    fields = list(rows[0])
    with path.open("w", encoding="utf-8-sig", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=fields)
        writer.writeheader()
        for row in rows:
            writer.writerow({
                key: json.dumps(value, ensure_ascii=False)
                if isinstance(value, (list, dict))
                else value
                for key, value in row.items()
            })

write_csv(output_dir / "add_k_tuning_grid.csv", add_k_grid_rows)
write_csv(output_dir / "interpolation_tuning_grid.csv", interpolation_grid_rows)
write_csv(output_dir / "probability_normalization_checks.csv", normalization_rows)

summary = {
    "model_directory": str(MODEL_DIR),
    "development_path": str(DEV_PATH),
    "development_tuning_only": True,
    "test_data_used": False,
    "perplexity_computed": False,
    "seed": SEED,
    "sampled_complete_sentences": sample_size,
    "scored_prediction_events_including_eos": event_count,
    "add_k_grid": ADD_K_GRID,
    "best_add_k_by_order": {
        str(order): best_k[order] for order in range(1, 5)
    },
    "interpolation_weight_grid_step": WEIGHT_STEP,
    "interpolation_candidates_tested": len(weight_candidates),
    "best_interpolation": best_interpolation,
    "prediction_vocabulary_size": len(model.allowed_outcomes),
    "normalization_tolerance": NORMALIZATION_TOLERANCE,
    "normalization_checks": normalization_rows,
}
summary_path = output_dir / "development_tuning_report.json"
summary_path.write_text(
    json.dumps(summary, ensure_ascii=False, indent=2, allow_nan=False) + "\n",
    encoding="utf-8",
)

print("Sampled complete dev sentences:", sample_size)
print("Scored dev prediction events including EOS:", event_count)
print("\nBest add-k by order (selection objective: mean dev log probability)")
for order in range(1, 5):
    print(f"order {order}: {best_k[order]}")
print("\nBest interpolation:")
print(best_interpolation)
print(
    f"\nNormalization checks passed: {len(normalization_rows)} distributions; "
    f"maximum error="
    f"{max(row['absolute_error_from_one'] for row in normalization_rows):.3g}"
)
print("\nSaved outputs:")
for path in sorted(output_dir.iterdir()):
    print(f"{path.name}: {path.stat().st_size:,} bytes")
print("Output directory:", output_dir)

Sampled complete dev sentences: 5000
Scored dev prediction events including EOS: 80507

Best add-k by order (selection objective: mean dev log probability)
order 1: {'order': 1, 'k': 0.01, 'mean_dev_log_probability': -7.800991356931869, 'prediction_vocabulary_size': 49998, 'events': 80507}
order 2: {'order': 2, 'k': 0.01, 'mean_dev_log_probability': -6.264504835116886, 'prediction_vocabulary_size': 49998, 'events': 80507}
order 3: {'order': 3, 'k': 0.01, 'mean_dev_log_probability': -7.738734359772477, 'prediction_vocabulary_size': 49998, 'events': 80507}
order 4: {'order': 4, 'k': 0.01, 'mean_dev_log_probability': -9.060943847809796, 'prediction_vocabulary_size': 49998, 'events': 80507}

Best interpolation:
{'weights_unigram_bigram_trigram_fourgram': [0.25, 0.25, 0.25, 0.25], 'mean_dev_log_probability': -5.725790547628856, 'events': 80507}

Normalization checks passed: 25 distributions; maximum error=1.11e-16

Saved outputs:
add_k_tuning_grid.csv: 989 bytes
bigram_conditional_probabili

In [14]:
from pathlib import Path
from datetime import datetime, timezone
import csv
import json
import math
import numpy as np

assert all(name in globals() for name in (
    "ngram_counts", "context_totals", "component_matrix",
    "event_count", "representative_histories", "model",
)), "Run the previous development-tuning cell first in this session."

REFINED_K_GRID = [
    0.00001, 0.00003, 0.0001, 0.0003,
    0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0,
]
WEIGHT_STEP = 0.1
TOLERANCE = 1e-6

# Refine add-k values for each order using the same fixed-seed dev sample.
add_k_rows = []
best_k_refined = {}

for order in range(1, 5):
    counts = ngram_counts[order]
    totals = context_totals[order]
    best = None

    for k in REFINED_K_GRID:
        mean_logprob = float(
            (np.log(counts + k)
             - np.log(totals + k * len(model.allowed_outcomes))).mean()
        )
        row = {
            "order": order,
            "k": k,
            "mean_dev_log_probability": mean_logprob,
            "events": event_count,
        }
        add_k_rows.append(row)
        if best is None or mean_logprob > best["mean_dev_log_probability"]:
            best = row

    best_k_refined[order] = best

# Refine the four-order interpolation grid to 0.1 increments.
units = round(1.0 / WEIGHT_STEP)
weight_candidates = []
for a in range(units + 1):
    for b in range(units - a + 1):
        for c in range(units - a - b + 1):
            d = units - a - b - c
            weight_candidates.append(tuple(x / units for x in (a, b, c, d)))

interpolation_rows = []
best_interpolation_refined = None

for weights in weight_candidates:
    mixed = component_matrix @ np.asarray(weights, dtype=np.float64)
    mean_logprob = (
        float("-inf")
        if np.any(mixed <= 0)
        else float(np.log(mixed).mean())
    )
    row = {
        "weights_unigram_bigram_trigram_fourgram": list(weights),
        "mean_dev_log_probability": mean_logprob,
        "events": event_count,
    }
    interpolation_rows.append(row)
    if (best_interpolation_refined is None
            or mean_logprob
            > best_interpolation_refined["mean_dev_log_probability"]):
        best_interpolation_refined = row

if best_interpolation_refined is None or not math.isfinite(
    best_interpolation_refined["mean_dev_log_probability"]
):
    raise RuntimeError("No finite refined interpolation candidate was found")

# Check selected add-k and interpolation distributions on the same five contexts.
normalization_rows = []
for context_number, history in enumerate(representative_histories, start=1):
    for order in range(1, 5):
        context = list(history[-(order - 1):]) if order > 1 else []
        k = best_k_refined[order]["k"]
        total = math.fsum(
            model.prob(token_id, context, method="add_k", k=k)
            for token_id in model.allowed_outcomes
        )
        error = abs(total - 1.0)
        assert error <= TOLERANCE, (context_number, order, total)
        normalization_rows.append({
            "context_number": context_number,
            "method": "add_k",
            "order": order,
            "k": k,
            "probability_sum": total,
            "absolute_error": error,
        })

    weights = best_interpolation_refined[
        "weights_unigram_bigram_trigram_fourgram"
    ]
    context = list(history)
    total = math.fsum(
        model.prob(
            token_id, context,
            method="interpolation",
            weights=weights,
        )
        for token_id in model.allowed_outcomes
    )
    error = abs(total - 1.0)
    assert error <= TOLERANCE, (context_number, "interpolation", total)
    normalization_rows.append({
        "context_number": context_number,
        "method": "interpolation",
        "order": 4,
        "weights": weights,
        "probability_sum": total,
        "absolute_error": error,
    })

output_dir_refined = Path("/kaggle/working/ngram_results/analysis")
output_dir_refined.mkdir(parents=True, exist_ok=True)

def write_rows(path, rows):
    fields = list(dict.fromkeys(
        key for row in rows for key in row
    ))
    with path.open("w", encoding="utf-8-sig", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=fields)
        writer.writeheader()
        for row in rows:
            writer.writerow({
                key: json.dumps(value, ensure_ascii=False)
                if isinstance(value, (list, dict))
                else value
                for key, value in row.items()
            })
write_rows(output_dir_refined / "add_k_refined_grid.csv", add_k_rows)
write_rows(output_dir_refined / "interpolation_refined_grid.csv", interpolation_rows)
write_rows(output_dir_refined / "normalization_checks.csv", normalization_rows)
write_rows(
    output_dir_refined / "add_k_tuning.csv",
    [best_k_refined[order] for order in range(1, 5)],
)
(output_dir_refined / "interpolation_tuning.json").write_text(
    json.dumps(best_interpolation_refined, ensure_ascii=False, indent=2, allow_nan=False) + "\n",
    encoding="utf-8",
)
normalization_report = {
    "development_data_only": True,
    "test_data_used": False,
    "checks": normalization_rows,
    "check_count": len(normalization_rows),
    "maximum_absolute_error": max(row["absolute_error"] for row in normalization_rows),
    "tolerance": TOLERANCE,
}
(output_dir_refined / "probability_normalization_checks.json").write_text(
    json.dumps(normalization_report, ensure_ascii=False, indent=2, allow_nan=False) + "\n",
    encoding="utf-8",
)

summary = {
    "development_tuning_only": True,
    "test_data_used": False,
    "perplexity_computed": False,
    "sample_seed": 42,
    "sampled_complete_sentences": 5000,
    "scored_events_including_eos": event_count,
    "add_k_grid": REFINED_K_GRID,
    "best_add_k_by_order": {
        str(order): best_k_refined[order] for order in range(1, 5)
    },
    "interpolation_weight_step": WEIGHT_STEP,
    "interpolation_candidates_tested": len(weight_candidates),
    "best_interpolation": best_interpolation_refined,
    "prediction_vocabulary_size": len(model.allowed_outcomes),
    "normalization_tolerance": TOLERANCE,
    "normalization_checks_passed": len(normalization_rows),
    "maximum_normalization_error": max(
        row["absolute_error"] for row in normalization_rows
    ),
}
summary_path = output_dir_refined / "development_tuning_refined_report.json"
summary_path.write_text(
    json.dumps(summary, ensure_ascii=False, indent=2, allow_nan=False) + "\n",
    encoding="utf-8",
)

print("Best refined add-k values:")
for order in range(1, 5):
    print(order, best_k_refined[order])
print("\nBest refined interpolation:")
print(best_interpolation_refined)
print(
    f"\nNormalization checks passed: {len(normalization_rows)}; "
    f"maximum error={summary['maximum_normalization_error']:.3g}"
)
print("\nSaved files:")
for path in sorted(output_dir_refined.iterdir()):
    print(f"{path.name}: {path.stat().st_size:,} bytes")
print("Output directory:", output_dir_refined)

Best refined add-k values:
1 {'order': 1, 'k': 1e-05, 'mean_dev_log_probability': -7.8009912205530005, 'events': 80507}
2 {'order': 2, 'k': 0.01, 'mean_dev_log_probability': -6.264504835116886, 'events': 80507}
3 {'order': 3, 'k': 0.001, 'mean_dev_log_probability': -7.518115702352875, 'events': 80507}
4 {'order': 4, 'k': 0.0003, 'mean_dev_log_probability': -8.72957872053161, 'events': 80507}

Best refined interpolation:
{'weights_unigram_bigram_trigram_fourgram': [0.2, 0.4, 0.1, 0.3], 'mean_dev_log_probability': -5.7100815854737395, 'events': 80507}

Normalization checks passed: 25; maximum error=2.22e-16

Saved files:
add_k_refined_grid.csv: 1,512 bytes
add_k_tuning.csv: 180 bytes
add_k_tuning_grid.csv: 989 bytes
bigram_conditional_probabilities.csv: 511 bytes
bigram_conditional_probabilities.json: 1,377 bytes
bigram_top10_all_tokens.csv: 379 bytes
bigram_top10_words_only.csv: 456 bytes
development_tuning_refined_report.json: 1,333 bytes
development_tuning_report.json: 8,585 bytes
fou

In [16]:
from pathlib import Path
from datetime import datetime, timezone
import csv
import hashlib
import json
import math
import os
import shutil
import tempfile
import zipfile

import numpy as np

CODE_ROOT = Path("/kaggle/working/code")
if str(CODE_ROOT) not in __import__("sys").path:
    __import__("sys").path.insert(0, str(CODE_ROOT))
from src.ngram_model import BOS, NGramModel

RESULTS_ROOT = Path("/kaggle/working/ngram_results")
MODEL_DIR = RESULTS_ROOT / "model"
WORK_DIR = RESULTS_ROOT / "work"
ANALYSIS_DIR = RESULTS_ROOT / "analysis"
REPORTS_DIR = RESULTS_ROOT / "reports"
FINAL_DIR = RESULTS_ROOT / "final_downloads"
TRAINING_REPORT = REPORTS_DIR / "training_report.json"
MODEL_ZIP = FINAL_DIR / "trained_model.zip"
REPORTS_ZIP = FINAL_DIR / "analysis_reports.zip"
GIB = 1024 ** 3

assert (MODEL_DIR / "metadata.json").is_file(), f"Missing model: {MODEL_DIR}"
assert (MODEL_DIR / "vocab.txt").is_file()
metadata = json.loads((MODEL_DIR / "metadata.json").read_text(encoding="utf-8"))
model = NGramModel.load(MODEL_DIR, mmap_mode="r")
assert model.max_order == 4
assert set(model.orders) == {1, 2, 3, 4}

# Verify occurrence and unique counts against the previously verified full run.
expected_occurrences = 64_875_025
expected_unique = {"1": 49_998, "2": 10_265_086, "3": 27_987_839, "4": 38_108_498}
actual_occurrences = {str(order): model.total_occurrences(order) for order in range(1, 5)}
actual_unique = {str(order): model.unique_ngrams(order) for order in range(1, 5)}
assert all(value == expected_occurrences for value in actual_occurrences.values()), actual_occurrences
assert actual_unique == expected_unique, actual_unique
assert metadata.get("raw_occurrences_by_order") == actual_occurrences
assert metadata.get("unique_ngrams_by_order") == actual_unique

# Exercise the loaded API, including a probability distribution normalization check.
query_context = [BOS]
query_word = model.word_to_id["<UNK>"]
query_probability = model.prob(query_word, query_context, method="add_k", k=1.0)
query_log_probability = model.logprob(query_word, query_context, method="add_k", k=1.0)
assert math.isfinite(query_probability) and 0.0 <= query_probability <= 1.0
assert math.isclose(query_log_probability, math.log(query_probability), rel_tol=1e-12)
predictions = model.predict_next("እኔ", k=5)
assert len(predictions) <= 5
assert all(math.isfinite(score) and 0.0 <= score <= 1.0 for _, score in predictions)
assert predictions == sorted(predictions, key=lambda item: (-item[1], item[0]))

# Verify all required reports exist before building the archives.
expected_analysis_files = [
    "unigram_top10_all_tokens.csv", "bigram_top10_all_tokens.csv",
    "trigram_top10_all_tokens.csv", "fourgram_top10_all_tokens.csv",
    "unigram_top10_words_only.csv", "bigram_top10_words_only.csv",
    "trigram_top10_words_only.csv", "fourgram_top10_words_only.csv",
    "unigram_top10_all_tokens_with_probability.csv",
    "bigram_top10_all_tokens_with_probability.csv",
    "trigram_top10_all_tokens_with_probability.csv",
    "fourgram_top10_all_tokens_with_probability.csv",
    "unigram_top10_words_only_with_probability.csv",
    "bigram_top10_words_only_with_probability.csv",
    "trigram_top10_words_only_with_probability.csv",
    "fourgram_top10_words_only_with_probability.csv",
    "bigram_conditional_probabilities.csv", "bigram_conditional_probabilities.json",
    "target_sentence_probabilities.csv", "target_sentence_probabilities.json",
    "add_k_tuning.csv", "add_k_refined_grid.csv",
    "interpolation_tuning.json", "interpolation_refined_grid.csv",
    "probability_normalization_checks.json", "normalization_checks.csv",
    "development_tuning_refined_report.json", "development_tuning_report.json",
    "top10_and_bigram_report.json",
]
missing_or_empty = [
    name for name in expected_analysis_files
    if not (ANALYSIS_DIR / name).is_file() or (ANALYSIS_DIR / name).stat().st_size == 0
]
assert not missing_or_empty, f"Missing or empty analysis outputs: {missing_or_empty}"
normalization_report = json.loads(
    (ANALYSIS_DIR / "probability_normalization_checks.json").read_text(encoding="utf-8")
)
normalization_checks = normalization_report["checks"]
assert normalization_checks
assert normalization_report["development_data_only"] is True
assert normalization_report["test_data_used"] is False
assert all(row["absolute_error"] <= 1e-6 for row in normalization_checks)

# Ensure a small training report is available even when packaging an already-saved model.
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
if not TRAINING_REPORT.is_file():
    training_report = {
        "status": "model_loaded; runtime metrics unavailable in this session",
        "model_metadata": metadata,
        "verified_occurrences_by_order": actual_occurrences,
        "verified_unique_ngrams_by_order": actual_unique,
        "note": "Reconstructed from saved model metadata; this cell did not run training.",
    }
    TRAINING_REPORT.write_text(
        json.dumps(training_report, ensure_ascii=False, indent=2, allow_nan=False) + "\n",
        encoding="utf-8",
    )
assert TRAINING_REPORT.stat().st_size > 0

# Keep storage requirements explicit; ZIP_STORED avoids expensive compression of large arrays.
model_files = sorted(path for path in MODEL_DIR.iterdir() if path.is_file())
required_model_files = {"metadata.json", "vocab.txt"}
for order in range(1, 5):
    required_model_files.update({
        f"order_{order}_keys.u64", f"order_{order}_counts.u64",
        f"order_{order}_context_keys.u64", f"order_{order}_context_totals.u64",
    })
assert required_model_files.issubset({path.name for path in model_files}), (
    "Model directory is missing NGramModel.load() artifacts",
    sorted(required_model_files - {path.name for path in model_files}),
)
model_bytes = sum(path.stat().st_size for path in model_files)
working_free = shutil.disk_usage(RESULTS_ROOT).free
temporary_free = shutil.disk_usage(tempfile.gettempdir()).free
assert working_free >= model_bytes + GIB, (
    f"Need about {model_bytes / GIB:.2f} GiB for one model archive plus 1 GiB reserve; "
    f"only {working_free / GIB:.2f} GiB is free in /kaggle/working."
)
assert temporary_free >= model_bytes + GIB, (
    f"Need about {model_bytes / GIB:.2f} GiB to extract and verify the model ZIP "
    f"plus 1 GiB reserve; only {temporary_free / GIB:.2f} GiB is free in temp storage."
)

FINAL_DIR.mkdir(parents=True, exist_ok=True)
def sha256_file(path, block_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as file:
        while block := file.read(block_size):
            digest.update(block)
    return digest.hexdigest()

def write_atomic_zip(destination, writer_function):
    temporary = destination.with_suffix(destination.suffix + ".tmp")
    temporary.unlink(missing_ok=True)
    try:
        writer_function(temporary)
        os.replace(temporary, destination)
    finally:
        temporary.unlink(missing_ok=True)

# Save a content manifest with the analysis archive. The external artifact
# manifest below will additionally contain both archive hashes.
content_manifest = {
    "model_files": {},
    "analysis_and_report_files": {},
}
for file_path in model_files:
    content_manifest["model_files"][file_path.name] = {
        "bytes": file_path.stat().st_size,
        "sha256": sha256_file(file_path),
    }
for file_path in [*(ANALYSIS_DIR / name for name in expected_analysis_files), TRAINING_REPORT]:
    content_manifest["analysis_and_report_files"][str(file_path.relative_to(RESULTS_ROOT))] = {
        "bytes": file_path.stat().st_size,
        "sha256": sha256_file(file_path),
    }
CONTENT_MANIFEST = REPORTS_DIR / "artifact_contents_sha256.json"
CONTENT_MANIFEST.write_text(
    json.dumps(content_manifest, ensure_ascii=False, indent=2, allow_nan=False) + "\n",
    encoding="utf-8",
)

# Archive members keep the requested model/... structure and include file hashes.
def create_model_zip(path):
    with zipfile.ZipFile(path, "w", compression=zipfile.ZIP_STORED) as archive:
        checksum_lines = []
        for file_path in model_files:
            archive.write(file_path, arcname=f"model/{file_path.name}")
            checksum_lines.append(f"{sha256_file(file_path)}  {file_path.name}")
        archive.writestr("model/SHA256SUMS.txt", "\n".join(checksum_lines) + "\n")

write_atomic_zip(MODEL_ZIP, create_model_zip)

# Include the complete current analysis folder and training report in one archive.
def create_analysis_zip(path):
    with zipfile.ZipFile(path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
        for file_path in sorted(p for p in ANALYSIS_DIR.iterdir() if p.is_file()):
            archive.write(file_path, arcname=f"analysis/{file_path.name}")
        archive.write(TRAINING_REPORT, arcname="reports/training_report.json")
        archive.write(CONTENT_MANIFEST, arcname="reports/artifact_contents_sha256.json")

write_atomic_zip(REPORTS_ZIP, create_analysis_zip)

# Check ZIP integrity and reload a model extracted from the actual archive.
assert MODEL_ZIP.is_file() and MODEL_ZIP.stat().st_size > 0
assert REPORTS_ZIP.is_file() and REPORTS_ZIP.stat().st_size > 0
with zipfile.ZipFile(MODEL_ZIP) as archive:
    assert archive.testzip() is None
    model_members = set(archive.namelist())
    assert {f"model/{name}" for name in required_model_files}.issubset(model_members)
with zipfile.ZipFile(REPORTS_ZIP) as archive:
    assert archive.testzip() is None
    report_members = set(archive.namelist())
    assert "reports/training_report.json" in report_members
    assert "reports/artifact_contents_sha256.json" in report_members
    assert all(f"analysis/{name}" in report_members for name in expected_analysis_files)

with tempfile.TemporaryDirectory(prefix="ngram_archive_reload_") as temp_dir:
    with zipfile.ZipFile(MODEL_ZIP) as archive:
        archive.extractall(temp_dir)
    extracted_model_dir = Path(temp_dir) / "model"
    extracted_model = NGramModel.load(extracted_model_dir, mmap_mode="r")
    checksum_lines = (extracted_model_dir / "SHA256SUMS.txt").read_text(encoding="utf-8").splitlines()
    for line in checksum_lines:
        expected_hash, filename = line.split("  ", 1)
        assert sha256_file(extracted_model_dir / filename) == expected_hash, filename
    assert extracted_model.max_order == 4
    assert {str(order): extracted_model.total_occurrences(order) for order in range(1, 5)} == actual_occurrences
    extracted_probability = extracted_model.prob(query_word, query_context, method="add_k", k=1.0)
    extracted_log_probability = extracted_model.logprob(query_word, query_context, method="add_k", k=1.0)
    assert math.isclose(extracted_probability, query_probability, rel_tol=1e-12)
    assert math.isclose(extracted_log_probability, query_log_probability, rel_tol=1e-12)
    assert extracted_model.predict_next("እኔ", k=5) == predictions

# Record byte sizes and SHA-256 digests for every archived file and both archives.
manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "model_directory": str(MODEL_DIR),
    "archive_files": {},
    "model_files": {},
    "analysis_and_report_files": {},
    "checks": {
        "orders": 4,
        "occurrences_by_order": actual_occurrences,
        "unique_ngrams_by_order": actual_unique,
        "model_reload_from_extracted_zip": True,
        "probability_and_logprob_query": True,
        "predict_next_query": True,
        "development_normalization_checks": len(normalization_checks),
        "maximum_normalization_error": normalization_report["maximum_absolute_error"],
    },
}
for file_path in model_files:
    manifest["model_files"][file_path.name] = {
        "bytes": file_path.stat().st_size,
        "sha256": sha256_file(file_path),
    }
for file_path in [*(ANALYSIS_DIR / name for name in expected_analysis_files), TRAINING_REPORT, CONTENT_MANIFEST]:
    manifest["analysis_and_report_files"][str(file_path.relative_to(RESULTS_ROOT))] = {
        "bytes": file_path.stat().st_size,
        "sha256": sha256_file(file_path),
    }
for archive_path in (MODEL_ZIP, REPORTS_ZIP):
    manifest["archive_files"][archive_path.name] = {
        "bytes": archive_path.stat().st_size,
        "sha256": sha256_file(archive_path),
    }
manifest_path = REPORTS_DIR / "artifact_manifest.json"
manifest_path.write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2, allow_nan=False) + "\n",
    encoding="utf-8",
)

print("Verified model occurrences:", actual_occurrences)
print("Verified unique N-grams:", actual_unique)
print("Model array bytes:", model_bytes)
print("Normalization distributions checked:", len(normalization_checks))
print("Maximum normalization error:", normalization_report["maximum_absolute_error"])
print("\nDownloadable archives:")
for archive_path in (MODEL_ZIP, REPORTS_ZIP):
    print(f"{archive_path} | {archive_path.stat().st_size:,} bytes | SHA-256 {sha256_file(archive_path)}")
print("\nClick to download from this active Kaggle session:")
try:
    from IPython.display import FileLink, display
    display(FileLink(str(MODEL_ZIP)))
    display(FileLink(str(REPORTS_ZIP)))
    display(FileLink(str(manifest_path)))
except ImportError:
    print("Model archive:", MODEL_ZIP)
    print("Reports archive:", REPORTS_ZIP)
    print("Artifact manifest:", manifest_path)
print("\nManifest:", manifest_path)
print("These files remain in temporary Kaggle working storage until downloaded or saved in a notebook output/dataset.")


Verified model occurrences: {'1': 64875025, '2': 64875025, '3': 64875025, '4': 64875025}
Verified unique N-grams: {'1': 49998, '2': 10265086, '3': 27987839, '4': 38108498}
Model array bytes: 1826030820
Normalization distributions checked: 25
Maximum normalization error: 2.220446049250313e-16

Downloadable archives:
/kaggle/working/ngram_results/final_downloads/trained_model.zip | 1,826,034,822 bytes | SHA-256 ee95bdb28e45278b369faec50c9c5c03647dde631f75de59fbcb84427bba7ab5
/kaggle/working/ngram_results/final_downloads/analysis_reports.zip | 20,755 bytes | SHA-256 c445937f1896dff09d9594ef3d1f42ce19f378b0cdc21796abe3614d2ec2d72a

Click to download from this active Kaggle session:


/kaggle/working/ngram_results/final_downloads/trained_model.zip

/kaggle/working/ngram_results/final_downloads/analysis_reports.zip

/kaggle/working/ngram_results/reports/artifact_manifest.json


Manifest: /kaggle/working/ngram_results/reports/artifact_manifest.json
These files remain in temporary Kaggle working storage until downloaded or saved in a notebook output/dataset.


In [21]:
import csv
import json

assert "normalization_rows" in globals()
assert "output_dir_refined" in globals()

# Give both add-k and interpolation rows the same columns.
columns = [
    "context_number", "method", "order", "k", "weights",
    "probability_sum", "absolute_error",
]
normalization_path = output_dir_refined / "normalization_checks.csv"

with normalization_path.open("w", encoding="utf-8-sig", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=columns)
    writer.writeheader()
    for row in normalization_rows:
        writer.writerow({
            **row,
            "weights": json.dumps(row["weights"], ensure_ascii=False)
            if "weights" in row else "",
        })

summary = {
    "development_tuning_only": True,
    "test_data_used": False,
    "perplexity_computed": False,
    "sample_seed": 42,
    "sampled_complete_sentences": 5000,
    "scored_events_including_eos": event_count,
    "add_k_grid": REFINED_K_GRID,
    "best_add_k_by_order": {
        str(order): best_k_refined[order] for order in range(1, 5)
    },
    "interpolation_weight_step": WEIGHT_STEP,
    "interpolation_candidates_tested": len(weight_candidates),
    "best_interpolation": best_interpolation_refined,
    "prediction_vocabulary_size": len(model.allowed_outcomes),
    "normalization_tolerance": TOLERANCE,
    "normalization_checks_passed": len(normalization_rows),
    "maximum_normalization_error": max(
        row["absolute_error"] for row in normalization_rows
    ),
}

summary_path = output_dir_refined / "development_tuning_refined_report.json"
summary_path.write_text(
    json.dumps(summary, ensure_ascii=False, indent=2, allow_nan=False) + "\n",
    encoding="utf-8",
)

print("Repair complete. Existing tuning CSVs were preserved.")
for path in sorted(output_dir_refined.iterdir()):
    print(f"{path.name}: exists={path.is_file()}, bytes={path.stat().st_size:,}")
print("Output directory:", output_dir_refined)

Repair complete. Existing tuning CSVs were preserved.
add_k_refined_grid.csv: exists=True, bytes=1,512
development_tuning_refined_report.json: exists=True, bytes=1,333
interpolation_refined_grid.csv: exists=True, bytes=13,169
normalization_checks.csv: exists=True, bytes=915
Output directory: /kaggle/working/ngram_results/member2_dev_tuning_refined_20261023T140801Z
